Show current python package

In [1]:
import sys
print(sys.executable)

/scratch/users/yinya/pixi-py/.pixi/envs/default/bin/python


## Import and Process Data

In [1]:
import uproot
import numpy as np
import pandas as pd

In [2]:
def split_constant_nonconstant_columns(df: pd.DataFrame, return_names = False):
    """
    Splits a DataFrame into:
    - constant_df: columns where all values are identical
    - nonconstant_df: columns where at least one value differs
    """
    
    # A column is constant if it has exactly one unique value
    constant_columns = df.nunique(dropna=False) == 1
    
    constant_df = df.loc[:, constant_columns]
    nonconstant_df = df.loc[:, ~constant_columns]
    
    if return_names:
        return constant_df.columns, nonconstant_df.columns
    else:
        return constant_df, nonconstant_df

Import Data

In [3]:
with uproot.open("ntuple.18192176.0.root") as file:
    key = file.keys()[0]
    tree = uproot.open("ntuple.18192176.0.root:" + key)
    print("File opened successfully!")
    print(f"Available classes: {file.classnames()}")
    print(f"Imported Class: {key}")
    print(f"Tree information: {tree}")

File opened successfully!
Available classes: {'susy;1': 'TTree'}
Imported Class: susy;1
Tree information: <TTree 'susy' (1536 branches) at 0x7feaac67fb90>


Select Branches

In [4]:
theory_param = [branch for branch in tree.keys() if branch.startswith('IN')]
measure_param = [branch for branch in tree.keys() if branch.startswith('SP')]

Build Trees and remove unnecessary branches (branch with constant values)

In [5]:
theory_tree = tree.arrays(theory_param, library='pd')
measure_tree = tree.arrays(measure_param, library='pd')
_, theory_param_nonconst = split_constant_nonconstant_columns(theory_tree, return_names=True)
_, measure_param_nonconst = split_constant_nonconstant_columns(measure_tree, return_names=True)
del theory_tree
del measure_tree

## Train Model

In [6]:
def to_binary(arr, threshold=1, equal=False):
    if equal:
        return (np.asarray(arr) >= threshold).astype(int)
    else:
        return (np.asarray(arr) > threshold).astype(int)

In [7]:
data = tree.arrays(theory_param_nonconst, library='pd')
label = to_binary(tree['MO_Omega'].array(), threshold=0.12)

In [11]:
label

array([1, 1, 0, ..., 0, 1, 1], shape=(10000,))

Check Device Availability

In [8]:
!nvidia-smi

Tue Apr 28 16:55:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.58.03              Driver Version: 595.58.03      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:01:00.0 Off |                    0 |
| N/A   32C    P0             52W /  400W |       4MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [10]:
!nvidia-smi

Mon Apr 13 14:10:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.58.03              Driver Version: 595.58.03      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:01:00.0 Off |                    0 |
| N/A   33C    P0             53W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [9]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

In [10]:
# Check CUDA is available
print(torch.cuda.is_available())
print(torch.cuda.current_device())


True
0


In [11]:
current_device_number = 0
torch.cuda.set_device(current_device_number)

In [12]:


print(torch.cuda.get_device_name(current_device_number))

NVIDIA A100-SXM4-40GB


In [13]:
# Define device based on availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [18]:
theory_param_nonconst

Index(['IN_M_1', 'IN_M_2', 'IN_M_3', 'IN_At', 'IN_Ab', 'IN_Atau', 'IN_mu',
       'IN_tanb', 'IN_mA', 'IN_mqL3', 'IN_mtR', 'IN_mbR'],
      dtype='object')

The following code is ai generated with a prompt requiring to convert my old sklearn model to a pytorch model.

Since the primary goal is to test GPU usage, I haven't read into the code yet.

In [ ]:
# ── Data preparation ──────────────────────────────────────────────
data_train, data_test, label_train, label_test = train_test_split(
    data, label, test_size=0.1, random_state=42
)

# Convert to PyTorch tensors and move to device
X_train = torch.FloatTensor(data_train.values).to(device) #adding .values to convert from DataFrame to numpy array
X_test  = torch.FloatTensor(data_test.values).to(device)
y_train = torch.LongTensor(label_train).to(device)
y_test  = torch.LongTensor(label_test).to(device)

# DataLoader for batching
train_dataset = TensorDataset(X_train, y_train)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True)

# ── Model definition ──────────────────────────────────────────────
# Mirrors MLPClassifier(hidden_layer_sizes=(100, 50), activation="relu")
n_features = data_train.shape[1]
n_classes  = len(set(label))

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_features, 100),
            nn.ReLU(),
            nn.Linear(100, 50),
            nn.ReLU(),
            nn.Linear(50, n_classes)
        )
    
    def forward(self, x):
        return self.layers(x)

model     = MLP().to(device)
criterion = nn.CrossEntropyLoss()                    # equivalent to sklearn's loss
optimizer = torch.optim.Adam(model.parameters())     # solver="adam"

# ── Training loop ─────────────────────────────────────────────────
# max_iter=500 in sklearn = 500 epochs
epochs = 500
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss   = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if epoch % 50 == 0:
        print(f"Epoch {epoch}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

# ── Evaluation ────────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    output      = model(X_test)
    predictions = output.argmax(dim=1)
    accuracy    = (predictions == y_test).float().mean()
    print(f"\nTest Accuracy: {accuracy:.4f}")

Epoch 0/500 | Loss: 14.8289
Epoch 50/500 | Loss: 0.1995
Epoch 100/500 | Loss: 0.1287
Epoch 150/500 | Loss: 0.0867
Epoch 200/500 | Loss: 0.0887
Epoch 250/500 | Loss: 0.1080
Epoch 300/500 | Loss: 0.0827
Epoch 350/500 | Loss: 0.1131
Epoch 400/500 | Loss: 0.0773
Epoch 450/500 | Loss: 0.1038

Test Accuracy: 0.9000


In [17]:
type(label_train)

numpy.ndarray